# Mock 5C · CarbonGlobe: 40-Year Forest Carbon Emulation · Mô phỏng rừng toàn cầu sau 40 năm (Time Series · Chuỗi thời gian)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | The *Ecosystem Demography* model (ED v3) simulates how a forest grows from the weather it receives, but it is slow to run over the whole globe. Build a fast **emulator**, as in the Kaggle *IEEE BigData Cup 2026 · AI Emulation Challenge*: for each forest **site** (a 0.5° ≈ 55 km grid cell) and each of the **15 seed forest ages** (1–500 years), start from the **initial state** (year 0 = 1980), use the **40 years** of monthly weather (1981–2020) and predict the **year-40 state** (2020) of tree **height** and above-ground biomass (**agb**). | Mô hình *Ecosystem Demography* (ED v3) mô phỏng rừng phát triển theo thời tiết, nhưng chạy cho cả địa cầu rất chậm. Hãy xây dựng một **mô hình thay thế (emulator)** nhanh, như cuộc thi Kaggle *IEEE BigData Cup 2026 · AI Emulation Challenge*: với mỗi **điểm rừng** (ô lưới 0,5° ≈ 55 km) và mỗi **tuổi rừng ban đầu** trong 15 tuổi (1–500 năm), xuất phát từ **trạng thái ban đầu** (năm 0 = 1980), dùng thời tiết từng tháng trong **40 năm** (1981–2020) để dự đoán **trạng thái năm thứ 40** (2020) của **chiều cao cây** (`height`) và **sinh khối trên mặt đất** (`agb`). |
| **Data** | `train/train.npz`: 5,000 sites × 15 seed ages with the **full** yearly trajectories 1980–2020 of 7 ED outputs. `public_test/` and `private_test/`: 1,000 / 1,000 other sites (the forcing and the initial state only). In both test sets half of the sites come from the **hottest climates** (mean air temperature ≥ 26 °C), which train does not cover: the stand-in for the competition's warmer future-climate scenarios. | `train/train.npz`: 5.000 điểm × 15 tuổi rừng với **đầy đủ** quỹ đạo hằng năm 1980–2020 của 7 đầu ra ED. `public_test/` và `private_test/`: 1.000 / 1.000 điểm khác (chỉ có thời tiết và trạng thái ban đầu). Trong cả hai tập test, một nửa số điểm thuộc **vùng khí hậu nóng nhất** (nhiệt độ không khí trung bình ≥ 26 °C), không có trong train: thay cho các kịch bản khí hậu tương lai nóng hơn của cuộc thi gốc. |
| **Metric** | **MSE** over the two columns, which are **divided by their global means** (height / 10.90645, agb / 3.39567): the mean squared *relative* error at year 40 (Kaggle's rule). **Lower is better.** Ranking uses the **private** score. | **MSE** trên hai cột, mỗi cột đã **chia cho giá trị trung bình toàn cầu** (height / 10.90645, agb / 3.39567): sai số tương đối bình phương trung bình ở năm 40 (luật của Kaggle). **Càng thấp càng tốt.** Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv`: one row per (site, seed age), `id = <site_id>_<age>`, columns `id,height,agb` — **already divided by the means** (an unscaled file scores as if completely wrong). | `public_submission.csv`, `private_submission.csv`: mỗi dòng là một (điểm, tuổi rừng), `id = <site_id>_<tuổi>`, các cột `id,height,agb` — **đã chia cho giá trị trung bình** (file chưa chia sẽ bị chấm như sai hoàn toàn). |
| **Rules** | PyTorch model, **no external data**. The original CarbonGlobe dataset is public and contains the answers: downloading or using it in any way breaks the honour code. | Mô hình PyTorch, **không dùng dữ liệu ngoài**. Bộ dữ liệu CarbonGlobe gốc là công khai và chứa đáp án: tải về hoặc dùng nó dưới bất kỳ hình thức nào là vi phạm quy tắc danh dự. |

$$\text{score}=\frac{1}{2}\left[\frac{1}{N}\sum_{i=1}^{N}\left(\frac{h_i-\hat h_i}{10.90645}\right)^2+\frac{1}{N}\sum_{i=1}^{N}\left(\frac{b_i-\hat b_i}{3.39567}\right)^2\right]$$

🇬🇧 $N$ = rows (sites × 15 ages), $h$ = height (m), $b$ = agb (kg C/m²) in 2020. $\sqrt{\text{score}}$ is the relative RMSE.
🇻🇳 $N$ = số dòng (điểm × 15 tuổi), $h$ = chiều cao (m), $b$ = sinh khối (kg C/m²) năm 2020. $\sqrt{\text{score}}$ là RMSE tương đối.

**Files · Các file** (`np.load(path)`, every array has a name · mỗi mảng có tên):
- `train.npz`: `forcing` (sites, 480 months, 14), `static` (sites, 9), `targets` (sites, 15 ages, 41 years 1980–2020, 7) — year 0 is the initial state, year 40 the answer · năm 0 là trạng thái ban đầu, năm 40 là đáp án; `ages`, `years`, `site_id`, `*_names`. All zeros = ED simulated no forest for that site and age · toàn số 0 = ED không mô phỏng được rừng.
- `public_test.npz`, `private_test.npz` (same layout as the competition's `test_x` / `test_y0` · cùng cấu trúc với cuộc thi): `site_id`, `forcing` (sites, 480, 14), `static` (sites, 9), `y0` (sites, 15 ages, 7 outputs: initial state · trạng thái ban đầu), `ages`
- Forcing (monthly, Jan 1981 – Dec 2020 · theo tháng): `t_air` (K), `precip`, `t_deep1..6` (temperatures that vary less and less, likely soil layers · nhiệt độ các lớp đất), `co2` (ppm), `q_air` (humidity · độ ẩm), `t_range` (day–night range · biên độ ngày đêm, K), `sw_mean`, `sw_max` (sunlight · bức xạ mặt trời, W/m²), `wind` (m/s)
- Static · tĩnh: `soil_1..7` (soil parameters · thông số đất), `lat`, `lon` (rounded to 1° · làm tròn đến 1°)
- Outputs · đầu ra: `height`, `agb` (scored · được chấm), `soil_c` (soil carbon · carbon trong đất), `lai` (leaf area index · chỉ số diện tích lá), `gpp`, `npp` (primary productivity · năng suất sơ cấp), `rh` (soil respiration · hô hấp dị dưỡng): free extra supervision · giám sát bổ sung miễn phí

🇬🇧 A stand of seed age $a$ is $a + k$ years old in year $k$. Young stands grow fast, old ones are near equilibrium; some lose biomass after bad years.
🇻🇳 Rừng có tuổi ban đầu $a$ sẽ $a + k$ tuổi ở năm thứ $k$. Rừng non tăng trưởng nhanh, rừng già gần trạng thái cân bằng; có rừng bị mất sinh khối sau những năm xấu.

Data · Dữ liệu: **CarbonGlobe** (Wang et al., *NeurIPS 2025 Datasets & Benchmarks*) as used in [IEEE BigData Cup 2026 · AI Emulation Challenge (Kaggle)](https://www.kaggle.com/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge) · [dataset](https://www.kaggle.com/datasets/zhihaow/carbonglobe) · [GitHub](https://github.com/zhwang0/carbon-globe), licence **CC BY 4.0** — see `dataset/README.md` for the changes made · xem các thay đổi trong `dataset/README.md`.

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "5C"
FOLDER  = "5C_carbon_globe"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/mock-5"

# Already have the folder? Point this at the parent of 5C_carbon_globe/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 5C_carbon_globe/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/5C_carbon_globe/dataset"
TRAIN_NPZ = f"{DATA_DIR}/train/train.npz"
PUBLIC_NPZ = f"{DATA_DIR}/public_test/public_test.npz"
PRIVATE_NPZ = f"{DATA_DIR}/private_test/private_test.npz"
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
VAL_FRACTION = 0.1  # share of train sites kept for validation / tỷ lệ điểm train dùng để kiểm tra
BATCH_SIZE = 512
EPOCHS = 15
LR = 1e-3
HIDDEN = 256
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Features · Đọc dữ liệu và tạo đặc trưng

In [ ]:
train = dict(np.load(TRAIN_NPZ))
public = dict(np.load(PUBLIC_NPZ))
private = dict(np.load(PRIVATE_NPZ))
TARGETS = [str(t) for t in train["target_names"]]
AGES = train["ages"]
# EN: the competition's global means: the answers are height / 10.90645 and agb / 3.39567
# VI: giá trị trung bình toàn cầu của cuộc thi: đáp án là height / 10.90645 và agb / 3.39567
MEANS = np.array([10.90645, 3.39567], dtype=np.float32)
print("train:", {k: v.shape for k, v in train.items() if v.ndim > 1})
print("public:", {k: v.shape for k, v in public.items() if v.ndim > 1})
print(f"Public sites: {len(public['site_id'])} | Private sites: {len(private['site_id'])} | rows = sites x 15 ages")
print("outputs:", TARGETS)

# EN: standardisation statistics from the training sites / VI: thống kê chuẩn hoá lấy từ các điểm train
F_MU = train["forcing"].reshape(-1, 14).mean(0)
F_SD = train["forcing"].reshape(-1, 14).std(0) + 1e-6
Y_SD = train["targets"].reshape(-1, 7).std(0) + 1e-6


def site_features(d):
    """EN: per site: static features + the mean weather of 1981-2000 and of 2001-2020 (monthly detail thrown away).
    VI: mỗi điểm: đặc trưng tĩnh + thời tiết trung bình 1981-2000 và 2001-2020 (bỏ chi tiết từng tháng)."""
    s = d["static"].copy()
    s[:, 0] = np.log10(s[:, 0])  # soil_1 spans 5,000 .. 4,000,000 / soil_1 trải từ 5.000 đến 4.000.000
    f = (d["forcing"] - F_MU) / F_SD
    return np.concatenate([s, f[:, :240].mean(1), f[:, 240:].mean(1)], 1)  # 240 months = 20 years


def make_features(d, y0):
    """EN: one row per (site, age): site features, the initial state (scaled) and the seed age.
    VI: mỗi dòng là một (điểm, tuổi): đặc trưng điểm, trạng thái ban đầu (đã chuẩn hoá) và tuổi rừng."""
    site = site_features(d)
    n = len(site)
    return np.concatenate([np.repeat(site, 15, axis=0), (y0 / Y_SD).reshape(n * 15, 7),
                           np.tile(AGES / 500.0, n)[:, None]], 1).astype(np.float32)


# EN: split by site, so that validation sites are unseen like the test sites
# VI: tách theo điểm để các điểm validation chưa từng thấy, giống tập test
rng = np.random.default_rng(SEED)
sites = rng.permutation(len(train["site_id"]))
n_val = int(VAL_FRACTION * len(sites))
va_idx, tr_idx = sites[:n_val], sites[n_val:]


def train_part(idx):
    d = {"static": train["static"][idx], "forcing": train["forcing"][idx]}
    x = make_features(d, train["targets"][idx, :, 0])                     # year 0 = initial state / trạng thái đầu
    y = (train["targets"][idx, :, 40, :2] / MEANS).reshape(-1, 2)         # year 40 height, agb / năm 40
    return x, y.astype(np.float32)


X_tr, y_tr = train_part(tr_idx)
X_va, y_va = train_part(va_idx)
X_MU, X_SD = X_tr.mean(0), X_tr.std(0) + 1e-6
norm = lambda x: torch.tensor((x - X_MU) / X_SD, dtype=torch.float32)  # noqa: E731
train_loader = DataLoader(TensorDataset(norm(X_tr), torch.tensor(y_tr)), batch_size=BATCH_SIZE, shuffle=True)
print(f"Training rows: {len(X_tr):,} | Validation rows: {len(X_va):,} | features: {X_tr.shape[1]}")

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
class CarbonMLP(nn.Module):
    """EN: flat features -> MLP -> year-40 height and agb (divided by the means).
    VI: vector đặc trưng -> MLP -> height và agb năm 40 (đã chia cho giá trị trung bình)."""

    def __init__(self, n_in, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 2))

    def forward(self, x):
        return self.net(x)


model = CarbonMLP(X_tr.shape[1]).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
def mse(y_true, y_pred):
    """EN: the competition metric on (n, 2) arrays already divided by the means. VI: độ đo của cuộc thi."""
    return float(((y_true - y_pred) ** 2).mean())


@torch.no_grad()
def predict(model, x):
    model.eval()
    return model(norm(x).to(DEVICE)).cpu().numpy()


y0_va = (train["targets"][va_idx, :, 0, :2] / MEANS).reshape(-1, 2)
print(f"Validation MSE of 'keep the initial state' / giữ nguyên trạng thái ban đầu: {mse(y_va, y0_va):.4f}")

criterion = nn.MSELoss()  # = the metric / chính là độ đo
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)

    val = mse(y_va, predict(model, X_va))
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(X_tr):.4f} | Val MSE={val:.4f}"
    if val < best_val:
        best_val = val
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation MSE: {best_val:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(d, preds, csv_path):
    """EN: one row per (site, seed age): id = <site_id>_<age>, height and agb divided by the means. Save + zip.
    VI: mỗi dòng là một (điểm, tuổi rừng): id = <site_id>_<tuổi>, height và agb đã chia cho trung bình. Lưu + nén."""
    df = pd.DataFrame(preds.reshape(-1, 2), columns=["height", "agb"])
    df.insert(0, "id", [f"{s}_{a}" for s in d["site_id"] for a in d["ages"]])
    df.to_csv(csv_path, index=False, float_format="%.6g")
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} ({len(df):,} rows) and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
for d, name in ((public, "public"), (private, "private")):
    export(d, predict(model, make_features(d, d["y0"])), f"{name}_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, d):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", "height", "agb"], f"{csv_path}: columns must be ['id', 'height', 'agb']"
    want = {f"{s}_{a}" for s in d["site_id"] for a in d["ages"]}
    assert len(sub) == len(want) and set(sub["id"]) == want, f"{csv_path}: ids do not match the test set"
    v = sub[["height", "agb"]].to_numpy(dtype=float)
    assert np.isfinite(v).all(), f"{csv_path}: values must be finite numbers / giá trị phải là số hữu hạn"
    assert np.abs(v).max() < 20, f"{csv_path}: values look unscaled - divide by the means / chưa chia cho trung bình?"


check_submission("public_submission.csv", public)
check_submission("private_submission.csv", private)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "5C"          # which problem this notebook is for
CONTEST = "mock-5"    # this notebook is for Mock Contest 5
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")